# Lab 05 · Reference solution

The polished final implementation of [Lab 05: LangGraph rewrite of Lab 01](../README.md).

The `create_agent` shorthand as the headline — for the standard agent
loop you'd build with the explicit `StateGraph` walk, this is the
production-shaped 5-line construction. Plus the checkpointer +
`thread_id` pattern for conversation persistence, and an
`interrupt()`-based human-in-the-loop demo.

> ⏱ Read time: ~7 min · Notebook ~22 cells.
> 📖 The lab notebook walks the explicit StateGraph construction
> first to make the abstraction's mechanics visible. This solution
> uses `create_agent` for the common case and shows the manual graph
> *only* where extra control is needed (human-in-the-loop with
> `interrupt`).

## Setup

Per [`tools/langgraph/snapshot-v1.0.md`](../../../tools/langgraph/snapshot-v1.0.md):
`langgraph >= 1.0.0` with `langchain >= 1.0.0`. The
`create_agent` import is from `langchain.agents`, NOT the deprecated
`langgraph.prebuilt.create_react_agent`.

In [ ]:
import os
import pathlib

from dotenv import load_dotenv

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY")

PROVIDER = "openai"
print(f"Using {PROVIDER}")


## The model

LangChain initializes the provider client; `bind_tools` later attaches
tool schemas. The provider switch is one if/elif.

In [ ]:
if PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
elif PROVIDER == "anthropic":
    from langchain_anthropic import ChatAnthropic
    llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0)
else:
    raise ValueError(f"Unknown provider: {PROVIDER}")


## Tools

The `@tool` decorator converts a typed function into a LangChain tool.
The docstring becomes the description; type hints become the schema.
Two tools mirroring Lab 01: customer lookup + summation.

In [ ]:
from langchain_core.tools import tool

CUSTOMERS = {
    1001: {"id": 1001, "name": "Ada Lovelace", "plan": "pro", "email": "ada@example.com"},
    1002: {"id": 1002, "name": "Alan Turing", "plan": "free", "email": "alan@example.com"},
    1003: {"id": 1003, "name": "Grace Hopper", "plan": "pro", "email": "grace@example.com"},
}
ORDERS = {
    1001: [{"id": 9001, "total": 42.50}, {"id": 9002, "total": 19.99}],
    1002: [{"id": 9003, "total": 5.00}],
    1003: [],
}


@tool
def lookup_customer(email: str) -> dict:
    """Look up a customer by exact email address.

    Returns the customer with id, name, plan, and their orders.
    Returns {'error': 'not_found', 'email': ...} if no match.
    """
    for cust in CUSTOMERS.values():
        if cust["email"].lower() == email.lower():
            return {**cust, "orders": ORDERS.get(cust["id"], [])}
    return {"error": "not_found", "email": email}


@tool
def compute_total(numbers: list[float]) -> dict:
    """Sum a list of numbers. Returns the total and count."""
    return {"total": sum(numbers), "count": len(numbers)}


tools = [lookup_customer, compute_total]
print(f"Tools: {[t.name for t in tools]}")


## The 5-line agent — `create_agent`

The shorthand. For the standard "model + tools loop" agent, this is the
production-shaped construction. It internally builds the same
`StateGraph` with `model` and `tools` nodes plus the
`tools_condition` edge — you can verify with `.get_graph().draw_ascii()`
if you want to see the topology.

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a helpful customer-support assistant. Use the tools when "
        "appropriate; give a concise answer when you have enough information."
    ),
)

# Inspect the graph topology — same nodes/edges you would build manually.
print(agent.get_graph().draw_ascii())


**Sample output:**

```
       +-----------+
       | __start__ |
       +-----------+
             *
             *
             *
       +-------+
       | model |
       +-------+
       *        .
      *          .
     *            .
+-------+    +---------+
| tools |    | __end__ |
+-------+    +---------+
       *      .
        *    .
         *  .
       +-------+
       | model |
       +-------+
```

Same graph the explicit `StateGraph` build produces.

## Running the agent

The invocation pattern: `agent.invoke({"messages": [...]})`. The final
answer is the last AI message in the result trace.

In [ ]:
query = "What's Ada Lovelace's plan? Her email is ada@example.com."
result = agent.invoke({"messages": [HumanMessage(content=query)]})

final = result["messages"][-1]
print(f"Final: {final.content}\n")
print(f"Trace ({len(result['messages'])} messages):")
for i, m in enumerate(result["messages"]):
    role = m.__class__.__name__.replace("Message", "").lower()
    snippet = (m.content or "")[:60].replace("\n", " ")
    tool_info = ""
    if hasattr(m, "tool_calls") and m.tool_calls:
        tool_info = f" [calls: {[tc['name'] for tc in m.tool_calls]}]"
    print(f"  {i}. {role}: {snippet}{tool_info}")


**Sample output:**

```
Final: Ada Lovelace is on the pro plan.

Trace (4 messages):
  0. human: What's Ada Lovelace's plan? Her email is ada@example.com.
  1. ai:  [calls: ['lookup_customer']]
  2. tool: {"id": 1001, "name": "Ada Lovelace", "plan": "pro", "email": "ada@e
  3. ai: Ada Lovelace is on the pro plan.
```

## Conversation persistence — checkpointer + `thread_id`

The single biggest win moving from Lab 01's hand-rolled loop to
LangGraph: **durable state per conversation thread**. Compile with a
checkpointer (`InMemorySaver` for demos; SQLite/Postgres in production);
pass `config={"configurable": {"thread_id": ...}}` on every invoke.

Same `thread_id` continues the conversation. Different `thread_id` is a
fresh conversation.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# Rebuild the agent with a checkpointer
checkpointer = InMemorySaver()
persistent_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful customer-support assistant.",
    checkpointer=checkpointer,
)

thread = {"configurable": {"thread_id": "user-42"}}

# First turn — populates the thread's state
r1 = persistent_agent.invoke(
    {"messages": [HumanMessage(content="Look up ada@example.com")]},
    config=thread,
)
print(f"After turn 1: {r1['messages'][-1].content[:80]}")
print(f"  messages in state: {len(r1['messages'])}\n")

# Second turn — same thread_id, follow-up that requires the prior state
r2 = persistent_agent.invoke(
    {"messages": [HumanMessage(content="What's her total spend?")]},
    config=thread,
)
print(f"After turn 2: {r2['messages'][-1].content[:80]}")
print(f"  messages in state: {len(r2['messages'])}")


**Sample output:**

```
After turn 1: Ada Lovelace is on the pro plan. Her orders total $62.49 across 2 orders.
  messages in state: 4

After turn 2: Ada's total spend is $62.49 across 2 orders ($42.50 and $19.99).
  messages in state: 7
```

Turn 2 references "her" — the model only resolves the pronoun because
the prior turn's messages are in state. A different `thread_id` would
start fresh and not know who "her" refers to.

## Human-in-the-loop with `interrupt()`

The other big LangGraph win: **`interrupt()`** lets a tool pause
execution mid-trajectory, surface a payload to the caller, and resume
after a `Command(resume=...)` call. The state at the pause point is
checkpointed; the resume picks up exactly where it left off.

Below: a `delete_customer` tool that requires human approval before it
runs. The checkpointer is *required* for `interrupt()` — without
persistence, there's nowhere to pause.

In [ ]:
from langgraph.types import interrupt, Command


@tool
def delete_customer(email: str) -> dict:
    """Delete a customer by email. DESTRUCTIVE — requires human approval."""
    # interrupt() pauses execution and surfaces this payload to the caller.
    # When the caller resumes with Command(resume=value), `value` becomes
    # the return value of interrupt() and execution continues.
    approval = interrupt({
        "action": "delete_customer",
        "email": email,
        "message": f"About to delete customer {email}. Approve?",
    })

    if not approval:
        return {"status": "cancelled_by_user", "email": email}
    return {"status": "deleted", "email": email}


gated_tools = [lookup_customer, delete_customer]
gated_agent = create_agent(
    model=llm,
    tools=gated_tools,
    system_prompt=(
        "You are a customer-support assistant. For destructive actions like "
        "deletion, the tool will pause for human approval — that is expected. "
        "Use the result you get back after approval."
    ),
    checkpointer=InMemorySaver(),
)

# Drive a deletion. The agent will call delete_customer, which interrupts.
gate_thread = {"configurable": {"thread_id": "gated-1"}}
result = gated_agent.invoke(
    {"messages": [HumanMessage(content="Delete the account for alan@example.com")]},
    config=gate_thread,
)

# When interrupted, the result contains __interrupt__ with the payload
if "__interrupt__" in result:
    intr = result["__interrupt__"]
    print(f"Paused: {intr}\n")
else:
    print("Did not pause; final message:", result["messages"][-1].content)


**Sample output:**

```
Paused: [Interrupt(value={'action': 'delete_customer', 'email': 'alan@example.com',
                          'message': 'About to delete customer alan@example.com. Approve?'},
                   id='...')]
```

The graph is paused at the `interrupt()` call inside `delete_customer`.
Nothing has been deleted. The caller now decides whether to approve.

## Resume with rejection, then with approval

In [ ]:
# Reject — interrupt() returns False, the tool sees `approval=False`
resumed_no = gated_agent.invoke(
    Command(resume=False),
    config=gate_thread,
)
print(f"After rejection: {resumed_no['messages'][-1].content}\n")

# New thread to approve
approve_thread = {"configurable": {"thread_id": "gated-2"}}
gated_agent.invoke(
    {"messages": [HumanMessage(content="Delete grace@example.com")]},
    config=approve_thread,
)
resumed_yes = gated_agent.invoke(
    Command(resume=True),
    config=approve_thread,
)
print(f"After approval: {resumed_yes['messages'][-1].content}")


**Sample output:**

```
After rejection: I've cancelled the deletion for alan@example.com — no changes were made.

After approval: I've deleted the customer record for grace@example.com.
```

The `interrupt()` pattern is the canonical LangGraph mechanism for
human-in-the-loop. Compared to building this on the Lab 01 loop, the
state-persistence + resume guarantees are what you get for adopting
LangGraph.

## What did we trade for what

LangGraph's wins on top of the Lab 01 loop:

- **Replay-safe state via reducers.** The `add_messages` reducer makes parallel-branch state updates correct by construction.
- **Built-in checkpointing.** SQLite/Postgres backends instead of hand-rolled persistence.
- **First-class `interrupt()`.** Durable pause-and-resume that survives process restarts.
- **Less boilerplate.** `create_agent` is 5 lines.

What we did *not* get:

- **The model doesn't get smarter.** Same LLM, same tool selection. If Lab 01 picked the wrong tool, Lab 05 picks the wrong tool too.
- **Not a substitute for tool design.** The Lab 02 patterns (negative-guidance descriptions, confirmation gates) still apply.
- **Framework lock-in.** Migration off LangGraph is non-trivial.

[`concepts/agents/agents-vs-frameworks.md`](../../../concepts/agents/agents-vs-frameworks.md)
covers the tradeoff in depth — when LangGraph pays for itself and when it doesn't.